## <img style="float: left; padding-right: 10px; width: 100px" src="https://raw.githubusercontent.com/sbildea/SEED/main/imagini/unitbv_mie.png"> <center>Știința Explorării și Exploatării Datelor - SEED1</center>


##    <center>Curs 5a: Regresia logistică, pragul și matricea de confuzie</center>
**Universitatea Transilvania**<br>
**SIIPA, Semestrul I**<br>
**Instructor:** Conf. dr. Teodor &Scedil;tefan B&icirc;ldea<br>
**Contact:** teodor.bildea@unitbv.ro<br>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sbildea/SEED/blob/main/cursuri/05-clasificare/curs-05a-regresie-logistica.ipynb)

In [ ]:
# Configurare curs SEED: rulați această celulă prima.
# Colab: descarcă materialele cursului și instalează pachetele lipsă (aprox. un minut).
# Local: găsește singură rădăcina repository-ului, oricare ar fi folderul curent al kernelului.
import os, sys
MODUL = "05-clasificare"
if "google.colab" in sys.modules:
    if not os.path.exists("/content/SEED"):
        !git clone -q --depth 1 https://github.com/sbildea/SEED.git /content/SEED
        !pip install -q -r /content/SEED/requirements.txt
    RADACINA = "/content/SEED"
else:
    RADACINA = os.path.abspath(".")
    while not os.path.isdir(os.path.join(RADACINA, "seed")) and os.path.dirname(RADACINA) != RADACINA:
        RADACINA = os.path.dirname(RADACINA)
    import importlib.util, socket
    lipsa = [p for p in ("pandas", "matplotlib", "seaborn", "sklearn") if importlib.util.find_spec(p) is None]
    if lipsa:
        print("Lipsesc pachetele", lipsa, "- în terminal: pip install -r requirements-local.txt")
    try:
        socket.create_connection(("raw.githubusercontent.com", 443), 2).close()
    except OSError:
        print("Fără internet: imaginile din text nu se văd. O singură dată, în terminal: python tools/imagini_offline.py")
os.chdir(os.path.join(RADACINA, "cursuri", MODUL))
sys.path.insert(0, RADACINA)

## Cum lucrăm azi: tu, agentul și notebook-ul

Acest curs se joacă în trei: **lecția** (acest notebook), **tu** și **agentul tău** (Gemini în Colab, un agent din VS Code sau din Antigravity). Agentul scrie codul; tu decizi ce se cere, de ce, și verifici dacă rezultatul are sens. Fiecare idee este un **capitol** cu aceiași pași:

| Pas | Ce faci |
|---|---|
| **Concept** | citești 2-3 propoziții |
| **Prezice** | spui (sau scrii) ce crezi că va ieși, *înainte* să rulezi |
| **Cere agentului** | formulezi cererea; apoi rulezi codul de referință și îl compari cu ce a scris agentul |
| **Verifică** | compari cu valorile de control |
| **Capcană** | vezi ce poate merge prost, chiar dacă codul rulează |
| **Pentru client** | spui rezultatul în limbaj de manager |

**Primer** (lipește-l agentului la începutul conversației; dacă ai pierdut conversația sau agentul a rămas fără tokeni, lipește primerul și *Jurnalul de sesiune* de la finalul notebook-ului unui alt agent și scrie „continuă de aici”):

```text
Rolul tău: ești partenerul meu de lucru la cursul SEED1 (analiză de date și învățare automată), modulul 5: clasificare (regresie logistică, prag de decizie, matricea de confuzie, kNN, arbori de decizie, random forest, gradient boosting, SVM). Eu sunt student și nu știu să programez în Python; eu decid ce se face și de ce, tu scrii codul, iar eu verific rezultatul.

Datele cursului:
- Se încarcă DOAR cu `from seed.date import incarca`. `incarca("lab-set1")[0]` întoarce tabelul clienților unui operator de telecomunicații (Telco Customer Churn); ținta este coloana `Churn` (Yes / No).
- Nu folosi căi de fișiere și nu descărca date de pe internet.
- Folosește pandas, matplotlib/seaborn și scikit-learn. Cod scurt, comentarii în română, un pas pe rând.

Reguli pentru acest modul (le aplici chiar dacă nu le cer eu):
1. Scoate ID-urile (`customerID`). Verifică tipurile: coloanele numerice stocate ca text (de exemplu `TotalCharges`) se convertesc, iar valorile goale se tratează explicit, cu o explicație.
2. Împarte train/test stratificat după țintă (20% test, `random_state=42`) ÎNAINTE de orice transformare învățată din date; pregătirea și modelul stau într-un `Pipeline`.
3. Compară întotdeauna cu modelul de referință (clasa majoritară). Raportează acuratețea pe antrenare și pe test, plus recall, precizie și AUC pe test pentru clasa „pleacă”; nu te opri la acuratețe.
4. Standardizează datele pentru regresia logistică, kNN și SVM; arborii și ansamblurile de arbori nu au nevoie.
5. Când compari modele, folosește aceeași împărțire și aceleași metrici pentru toate, într-un singur tabel.
6. Nu inventa coloane și spune-mi dacă ceva pare ciudat (acuratețe aproape perfectă pe antrenare, un model sub cel de referință), în loc să-l prezinți ca pe un succes.

Cum lucrăm:
- Implicit ești în MOD TUTORE: înainte de a-mi da codul, îmi pui o singură întrebare scurtă despre concept („ce crezi că se va întâmpla?”). Nu rezolvi temele în locul meu; mă ajuți să verific.
- La cererea mea poți trece în MOD EXECUTANT (execuți direct și spui pe scurt ce ai făcut) sau în MOD RECENZENT (cauți greșeli în ce îți arăt: ID-uri printre predictori, evaluare pe datele de antrenare, transformări învățate înainte de împărțire).
- Când scriu „închide capitolul”, completezi Jurnalul de sesiune în formatul de mai jos, numai din ce am discutat. Dacă ceva nu s-a discutat, scrii „nu știu”.

Formatul Jurnalului de sesiune:
## Jurnal de sesiune
- Modul / capitolul la care am ajuns:
- Agent folosit și data:
- Ce am decis și de ce:
- Valori de control verificate (da / nu / care):
- Greșeli găsite la agent:
- Întrebare deschisă:
- Pentru următorul agent (ce a rămas de făcut, într-o propoziție):

Răspunde acum doar cu: rolul tău în două propoziții și cele 6 reguli, pe scurt.
```

## Problema de azi

Managerul de retenție al unui operator de telecomunicații: *„Care clienți urmează să plece, ca să le facem o ofertă înainte?”*

Ținta este acum o **categorie** (pleacă / rămâne), nu un număr: **clasificare**. **Proiectul vostru** este tocmai un clasificator pe setul vostru de date, cu **cel puțin trei modele comparate**. Azi construim protocolul și primul model (regresia logistică). În `curs-05b` vedem cum decid celelalte modele și le punem pe toate în aceeași **arenă**. Antrenarea pe setul proiectului o faceți la seminar.

Datele: `lab-set1` (Telco Customer Churn: 7.043 de clienți, 21 de coloane). Același client revine în modulul 6, unde transformăm greșelile modelului în bani.

**Planul de azi (≈55 min):** capitolul 0 cererea vagă · 1 acuratețea care minte · 2 de la dreaptă la probabilitate · 3 matricea de confuzie · 4 pragul și curba ROC · ce îi spunem clientului. Definițiile complete: secțiunea **Referință** de la sfârșit.

## Capitolul 0. Cererea vagă

**Concept.** Înainte de orice model, datele trebuie să fie ce par. Un agent grăbit ia doar coloanele numerice (și pierde informație) sau încearcă să transforme tot în numere și se oprește la prima eroare, „rezolvând-o” cum crede el.

**Prezice.** Câte coloane numerice crezi că are un tabel cu facturi lunare și totale, vechime în luni și vârstă?

**Cere agentului** (mod executant):

> Mod executant: încarcă `lab-set1` și spune-mi forma tabelului, câte coloane are de fiecare tip, ce tip are coloana `TotalCharges` și de ce, și dacă există o coloană de tip ID.

In [ ]:
# Cod de referință: ce primește un agent grăbit
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from seed.date import incarca

date = incarca("lab-set1")[0]
print("Forma:", date.shape)
print(date.dtypes.astype(str).value_counts().to_string())
goale = date["TotalCharges"].str.strip() == ""
print("\nTotalCharges are tipul", date["TotalCharges"].dtype, "| valori goale:", goale.sum(),
      "| vechimea (tenure) acestor clienți:", date.loc[goale, "tenure"].unique())
print("customerID: valori distincte", date["customerID"].nunique(), "din", len(date))
date.head(3)

**Verifică.** (7043, 21); doar 3 (`SeniorCitizen`, `tenure`, `MonthlyCharges`) coloane numerice; `TotalCharges` e text, cu 11 valori goale, toate la clienți cu vechime 0; `customerID` are 7.043 de valori distincte (câte una pe client).

**Capcană.** `TotalCharges` *arată* ca un număr, dar e text din cauza a 11 spații goale. Sunt clienți abia veniți (vechime 0), care încă n-au plătit nimic: valoarea corectă e 0, nu media. Un agent care completează golurile cu media le inventează plăți. Iar `customerID` e unic pentru fiecare client: un model îl poate memora, dar nu spune nimic despre un client nou.

In [ ]:
# Cod de referință: pregătirea minimă
df = date.drop(columns=["customerID"])                         # ID: nu spune nimic despre un client nou
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0)   # clienți noi: n-au plătit încă
y = (df.pop("Churn") == "Yes").astype(int)                     # ținta: 1 = pleacă, 0 = rămâne
X = df
print("Predictori:", X.shape, "  rata de plecare:", round(y.mean(), 3))

**Pentru client.** „Avem 7.043 de clienți și 19 caracteristici utile pentru fiecare; aproximativ unul din patru a plecat.”

## Capitolul 1. Acuratețea care minte

**Concept.** **Acuratețea** este proporția predicțiilor corecte. Când clasele sunt dezechilibrate (aici ~26,5% pleacă), un „model” care spune mereu „rămâne” are acuratețe mare și nu prinde pe nimeni. De aceea:
- împărțim **stratificat**: aceeași proporție de plecări în antrenare și în test (`stratify=y`);
- comparăm cu un **model de referință**: clasa cea mai frecventă (`DummyClassifier`), echivalentul „mediei” din modulul 4.

**Prezice.** Ce acuratețe are un model care spune despre *toți* clienții că rămân? Câți dintre cei care pleacă prinde?

**Cere agentului:**

> Împarte datele 80/20, stratificat după țintă (`random_state=42`). Construiește modelul de referință care prezice mereu clasa cea mai frecventă și raportează pe test acuratețea și câți dintre clienții care pleacă sunt prinși (recall).

In [ ]:
# Cod de referință: împărțirea stratificată și modelul de referință
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, recall_score, precision_score

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print("Rata de plecare:  antrenare", round(y_train.mean(), 3), "  test", round(y_test.mean(), 3))

referinta = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
p = referinta.predict(X_test)
print(f"Referința («nimeni nu pleacă»): acuratețe {accuracy_score(y_test, p):.3f}   plecări prinse (recall): {recall_score(y_test, p):.0%}")

**Verifică.** Rata de plecare 0,265 în antrenare și 0,265 în test; referința are acuratețe 0,735 și recall 0%.

**Capcană.** 73,5% acuratețe fără să prindă **niciun** client care pleacă. Orice „acuratețe de 80%” raportată de agent se citește față de 73,5%, nu față de 50%. Pe seturile dezechilibrate ale proiectelor (de exemplu boli rare, fraude), diferența e și mai mare.

**Pentru client.** „Dacă presupunem că nu pleacă nimeni, avem dreptate în 73,5% din cazuri, dar pierdem toți clienții care pleacă. Un model e util doar dacă îi găsește pe aceștia.”

## Capitolul 2. De la dreaptă la probabilitate

**Concept.** Am putea codifica „pleacă” = 1, „rămâne” = 0 și folosi regresia liniară din modulul 4. Dar o dreaptă urcă sau coboară la nesfârșit și poate da „probabilități” sub 0 sau peste 1. **Regresia logistică** trece aceeași combinație liniară prin funcția **sigmoidă**, care o strânge între 0 și 1:

$$P(\text{pleacă} \mid x) = \sigma(b_0 + b_1 x) = \frac{1}{1 + e^{-(b_0 + b_1 x)}}$$

Coeficienții nu se mai citesc „în unități ale țintei”: un $b_1$ negativ spune că probabilitatea **scade** când $x$ crește (cu cât, în Referința R1). Modelul dă o **probabilitate**; o transformăm în clasă cu un **prag** (capitolul 4).

**Prezice.** Cum crezi că variază probabilitatea de plecare cu vechimea clientului (`tenure`, în luni)? Ce ar prezice o dreaptă pentru un client cu 100 de luni?

**Cere agentului:**

> Pe antrenare, potrivește o regresie liniară și o regresie logistică pentru țintă în funcție de `tenure`. Desenează pe același grafic rata reală de plecare (pe grupe de 6 luni), dreapta și curba logistică, pentru vechimi de la 0 la 100 de luni.

In [ ]:
# Cod de referință: dreapta vs. sigmoida, cu un singur predictor
from sklearn.linear_model import LinearRegression, LogisticRegression

dreapta = LinearRegression().fit(X_train[["tenure"]], y_train)
sigmoida = LogisticRegression().fit(X_train[["tenure"]], y_train)
luni = pd.DataFrame({"tenure": np.arange(0, 101)})
rata = y_train.groupby(X_train["tenure"] // 6 * 6 + 3).mean()       # rata reală, pe grupe de 6 luni

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(rata.index, rata.values, color="k", s=15, label="rata reală de plecare (grupe de 6 luni)")
ax.plot(luni["tenure"], dreapta.predict(luni), label="regresie liniară")
ax.plot(luni["tenure"], sigmoida.predict_proba(luni)[:, 1], label="regresie logistică")
ax.axhline(0, color="grey", lw=0.8)
ax.axvline(72, color="grey", ls=":", lw=0.8)
ax.set_xlabel("vechime (luni); linia punctată = maximul din date")
ax.set_ylabel("probabilitatea de plecare")
ax.set_title("Dreapta coboară sub 0; sigmoida rămâne între 0 și 1")
ax.legend()
plt.tight_layout()
plt.show()

for t in [0, 72, 100]:
    x = pd.DataFrame({"tenure": [t]})
    print(f"vechime {t:3d} luni:   dreapta {dreapta.predict(x)[0]:+.3f}   logistică {sigmoida.predict_proba(x)[0, 1]:.3f}")
print(f"\nRegresia logistică: b0 = {sigmoida.intercept_[0]:.3f}, b1 = {sigmoida.coef_[0, 0]:.4f}")

**Verifică.** La vechime 0: dreapta +0,467, logistica 0,502; la 100 de luni: dreapta −0,154 (o „probabilitate” negativă), logistica 0,023. $b_1$ ≈ −0,038 (negativ: clienții vechi pleacă mai rar).

**Capcană.** Un client cu 100 de luni nu există încă în date (maximul e 72), dar va exista. Dreapta îi dă o probabilitate negativă; sigmoida nu iese niciodată din intervalul [0, 1]. Dacă agentul folosește `LinearRegression` pentru o țintă 0/1, e greșit chiar dacă scorul pare bun.

**Pentru client.** „Riscul de plecare scade puternic în primele luni de abonament: clienții noi sunt cei mai expuși.”

## Capitolul 3. Toate coloanele și matricea de confuzie

**Concept.** Cu toate coloanele, regresia logistică devine un model adevărat. Pregătirea este **recuzita din modulul 4**, neschimbată: `Pipeline` cu completarea lipsurilor, one-hot și standardizare. Predicțiile se compară cu realitatea în **matricea de confuzie**:

| | prezis: rămâne | prezis: pleacă |
|---|---|---|
| **real: rămâne** | adevărat negativ (TN) | **fals pozitiv** (FP): ofertă trimisă degeaba |
| **real: pleacă** | **fals negativ** (FN): client pierdut | adevărat pozitiv (TP) |

**Recall** $= \frac{TP}{TP + FN}$: din cei care pleacă, pe câți îi prindem. **Precizie** $= \frac{TP}{TP + FP}$: din cei pe care îi contactăm, câți chiar voiau să plece.

**Prezice.** Modelul va avea acuratețe peste 80%? Câți dintre cei care pleacă va rata?

**Cere agentului:**

> Antrenează o regresie logistică pe toate coloanele, într-un `Pipeline` cu completarea lipsurilor, one-hot și standardizare. Pe test, afișează acuratețea, recall-ul și precizia pentru clasa „pleacă” și matricea de confuzie cu etichete în română.

In [ ]:
# Recuzita din modulul 4, neschimbată: pregătirea datelor ca linie de procesare
from sklearn.compose import make_column_transformer, make_column_selector
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

def pregatire(scalare=False):
    """Rețeta de pregătire, învățată DOAR pe antrenare (în Pipeline).
    Numere: mediana în locul valorilor lipsă. Text: „Lipsă” în locul valorilor lipsă, apoi one-hot
    (categoriile noi din test sunt ignorate). La cerere: standardizarea tuturor coloanelor rezultate."""
    text = make_pipeline(SimpleImputer(strategy="constant", fill_value="Lipsă"),
                         OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    coloane = make_column_transformer((SimpleImputer(strategy="median"), make_column_selector(dtype_include="number")),
                                      (text, make_column_selector(dtype_exclude="number")),
                                      verbose_feature_names_out=False)
    return make_pipeline(coloane, StandardScaler()) if scalare else coloane

In [ ]:
# Cod de referință: regresia logistică pe toate coloanele
from sklearn.metrics import ConfusionMatrixDisplay

logistica = make_pipeline(pregatire(scalare=True), LogisticRegression(max_iter=1000)).fit(X_train, y_train)
p = logistica.predict(X_test)
print(f"Acuratețe {accuracy_score(y_test, p):.3f}   recall {recall_score(y_test, p):.3f}   precizie {precision_score(y_test, p):.3f}")

ConfusionMatrixDisplay.from_predictions(y_test, p, display_labels=["rămâne", "pleacă"], cmap="Blues", colorbar=False)
plt.title("Regresia logistică, prag 0,5 (test)")
plt.show()

**Verifică.** Acuratețe ≈ 0,807, recall ≈ 0,567, precizie ≈ 0,658. În matrice: 212 plecări prinse (TP), 162 ratate (FN), 110 oferte trimise degeaba (FP).

**Capcană.** Acuratețea a urcat doar de la 0,735 (referința) la 0,807, iar modelul ratează aproape 4 din 10 clienți care pleacă. Dacă agentul raportează doar acuratețea, nu vezi asta. Cere mereu matricea de confuzie și recall-ul clasei care te interesează.

**Pentru client.** „Din 374 clienți de test care au plecat, modelul i-ar fi semnalat pe 212; ar fi trimis și 110 oferte unor clienți care oricum rămâneau.”

## Capitolul 4. Pragul și curba ROC

**Concept.** Regresia logistică dă o probabilitate; `predict` o transformă în clasă cu **pragul 0,5**. Pragul este o **decizie de business**, nu o proprietate a modelului: dacă un client pierdut costă mult mai mult decât o ofertă trimisă degeaba, coborâm pragul și contactăm mai mulți.

**Curba ROC** arată toate pragurile deodată: pe orizontală rata falșilor pozitivi, pe verticală recall-ul. **AUC** (aria de sub curbă) este o singură notă, independentă de prag: 0,5 = ghicit, 1 = perfect. Interpretare: probabilitatea ca un client care pleacă, ales la întâmplare, să primească un scor mai mare decât unul care rămâne. AUC compară modele fără să alegem un prag; o folosim în arena din `curs-05b`.

**Prezice.** Dacă pragul coboară de la 0,5 la 0,3, cum se schimbă recall-ul? Dar precizia?

**Cere agentului:**

> Pentru regresia logistică de mai sus, fă un tabel pentru pragurile 0,5, 0,4, 0,3 și 0,2: câți clienți contactăm, câte plecări prindem, recall și precizie. Apoi desenează curba ROC pe test și calculează AUC.

In [ ]:
# Cod de referință: același model, praguri diferite
from sklearn.metrics import roc_auc_score, RocCurveDisplay

prob = logistica.predict_proba(X_test)[:, 1]          # probabilitatea de plecare
randuri = []
for prag in [0.5, 0.4, 0.3, 0.2]:
    p = (prob >= prag).astype(int)
    randuri.append({"prag": prag, "clienți contactați": int(p.sum()),
                    "plecări prinse": int(((p == 1) & (y_test == 1)).sum()), "din": int(y_test.sum()),
                    "recall": round(recall_score(y_test, p), 3), "precizie": round(precision_score(y_test, p), 3)})
praguri = pd.DataFrame(randuri)
print(praguri.to_string(index=False))

auc = roc_auc_score(y_test, prob)
RocCurveDisplay.from_predictions(y_test, prob, name=f"regresie logistică (AUC = {auc:.3f})")
plt.plot([0, 1], [0, 1], "k--", lw=0.8, label="ghicit (AUC = 0,5)")
plt.legend()
plt.title("Curba ROC: toate pragurile deodată")
plt.show()

**Verifică.** La pragul 0,5: 322 clienți contactați, recall 0,567, precizie 0,658. La 0,3: 544 contactați, recall 0,754, precizie 0,518. AUC ≈ 0,842.

**Capcană.** Coborând pragul prindem mult mai mulți clienți care pleacă, dar contactăm și mulți care ar fi rămas. Ce prag e „cel mai bun” nu o spune modelul, ci costurile (cât costă o ofertă, cât valorează un client păstrat). În modulul 6 îl calculăm în bani. Un agent care lasă implicit 0,5 a luat, fără să-ți spună, o decizie de business.

**Pentru client.** „Putem regla câți clienți contactăm: la pragul 0,3 îi prindem pe 75% din plecări, dar doar 52% din cei contactați ar fi plecat cu adevărat.”

## Ce îi spunem clientului

**Cere agentului:** „Scrie 3 propoziții pentru managerul de retenție, cu cifrele din notebook: cât de bine prinde modelul clienții care pleacă, ce înseamnă alegerea pragului și ce limită are analiza.”

Rezumatul de mai jos este calculat din variabilele notebook-ului; compară-l cu ce a scris agentul.

In [ ]:
# Rezumat construit din rezultatele de mai sus
p05 = praguri.set_index("prag").loc[0.5]
p03 = praguri.set_index("prag").loc[0.3]
print(f"1. Din {len(y_test)} clienți de test, {int(y_test.sum())} au plecat; a spune «nimeni nu pleacă» are acuratețe {1 - y_test.mean():.0%}, dar nu prinde pe nimeni.")
print(f"2. Regresia logistică, la pragul 0,5, prinde {p05['recall']:.0%} din plecări contactând {int(p05['clienți contactați'])} clienți; "
      f"la pragul 0,3 prinde {p03['recall']:.0%}, contactând {int(p03['clienți contactați'])}.")
print(f"3. AUC {auc:.2f}: modelul ordonează bine clienții după risc. Pragul se alege din costul ofertei și valoarea unui client (modulul 6).")

---
## Ce urmează

Avem protocolul (pregătire, împărțire stratificată, referință, matrice de confuzie, AUC) și primul model. Proiectul cere cel puțin trei. În `curs-05b` vedem **cum decid** kNN, arborii, random forest, gradient boosting și SVM și îi punem pe toți în aceeași arenă.

Continuăm cu [curs-05b-clasificare.ipynb](curs-05b-clasificare.ipynb). Puntea curs → seminar este la finalul lui.

---
# Referință

Definiții și detalii pentru acasă. Nu se parcurg la curs.

## R1. Sigmoida, șansele și coeficienții

Dacă $p = P(\text{pleacă})$, **șansele** (*odds*) sunt $\frac{p}{1-p}$ (de exemplu $p = 0{,}2$ înseamnă șanse de 1 la 4). Regresia logistică este o regresie liniară pentru logaritmul șanselor:

$$\ln \frac{p}{1-p} = b_0 + b_1 x_1 + \dots + b_k x_k$$

Când $x_j$ crește cu o unitate (ceilalți predictori rămânând la fel), șansele se înmulțesc cu $e^{b_j}$. Exemplu: $b_j = -0{,}04$ pentru vechime înseamnă că fiecare lună în plus înmulțește șansele de plecare cu $e^{-0{,}04} \approx 0{,}96$ (scad cu ~4%). Pe date standardizate, „o unitate” înseamnă o abatere standard.

Coeficienții se estimează prin **verosimilitate maximă** (fără formulă închisă ca la regresia liniară): se caută coeficienții pentru care datele observate sunt cele mai probabile.

## R2. Măsuri pentru clasificare

| Măsura | Formula | Ce spune |
|---|---|---|
| Acuratețe | $\frac{TP + TN}{n}$ | proporția predicțiilor corecte; înșelătoare la clase dezechilibrate |
| Recall (sensibilitate) | $\frac{TP}{TP + FN}$ | din cei care sunt pozitivi, câți sunt prinși |
| Precizie | $\frac{TP}{TP + FP}$ | din cei semnalați, câți sunt cu adevărat pozitivi |
| F1 | $2 \cdot \frac{\text{precizie} \cdot \text{recall}}{\text{precizie} + \text{recall}}$ | un compromis între cele două |
| AUC | aria de sub curba ROC | calitatea ordonării după scor, pentru toate pragurile; 0,5 = ghicit |

`classification_report` din scikit-learn le afișează pe toate, pe clase.

## R3. Regularizarea în regresia logistică

`LogisticRegression` este regularizată implicit (Ridge, L2). Intensitatea se dă prin `C` $= 1/\alpha$: **C mic = regularizare puternică**. Tipul se alege cu `l1_ratio` (scikit-learn ≥ 1.8): 0 = Ridge (L2), 1 = Lasso (L1), între = Elastic Net; pentru L1 sau Elastic Net se folosește `solver="saga"`. Ca în modulul 4, datele trebuie standardizate, iar `C` se alege prin validare încrucișată (`LogisticRegressionCV`).

## R4. Mai mult de două clase

Pentru o țintă cu mai multe clase (de exemplu calitatea unui vin: slabă / medie / bună), `LogisticRegression` estimează câte o probabilitate pentru fiecare clasă (regresie **multinomială**); predicția este clasa cu probabilitatea cea mai mare. Matricea de confuzie devine $k \times k$, iar recall-ul și precizia se calculează pe fiecare clasă.

## R5. Pentru aprofundare

- [extra-05-logistica-heart.ipynb](extra-05-logistica-heart.ipynb): regresia logistică pe date despre boli de inimă: animația sigmoidei, predictori categoriali, termeni de interacțiune, efectul lui `C`. Materialul din anii anteriori.
- [extra-05-exercitii-regresie-logistica.ipynb](extra-05-exercitii-regresie-logistica.ipynb): exerciții suplimentare.

## Jurnal de sesiune

Completează la sfârșit (sau cere agentului: *„închide capitolul”*) și păstrează-l în copia ta a notebook-ului. Dacă agentul rămâne fără tokeni, lipește primerul și acest jurnal unui alt agent și scrie „continuă de aici”.

- Modul / capitolul la care am ajuns:
- Agent folosit și data:
- Ce am decis și de ce:
- Valori de control verificate (da / nu / care):
- Greșeli găsite la agent:
- Întrebare deschisă:
- Pentru următorul agent (ce a rămas de făcut, într-o propoziție):